# Load the CIC UNSW-NB15 dataset into Neo4j

Dataset: **CIC-UNSW-NB15** (Canadian Institute for Cybersecurity re-extraction of the
UNSW-NB15 intrusion-detection dataset) - <https://www.unb.ca/cic/datasets/cic-unsw-nb15.html>.

The files under `.data/CIC-UNSW/` are:

| File | Rows | What it is |
|------|-----:|------------|
| `CICFlowMeter_out.csv` | 3,540,241 | **Full flow export** - the 5-tuple identity (IPs, ports, protocol), a timestamp, 76 CICFlowMeter statistical features, and the textual attack label. *This is what we load.* |
| `Data.csv` | 447,915 | The same 76 features, ML-ready, **without** identity columns. |
| `Label.csv` | 447,915 | Numeric label (0–9) row-aligned to `Data.csv`. |
| `Readme.txt` | - | The label code → name mapping. |

Only `CICFlowMeter_out.csv` carries the source/destination identities, so it is the only file from which a meaningful **graph** can be built. `DATASET_PATH` in `.env` points at it.

In [ ]:
# Load .env file
from dotenv import load_dotenv
import os

load_dotenv()
DATASET_PATH = os.getenv("DATASET_PATH")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

print("dataset :", DATASET_PATH)
print("neo4j   :", NEO4J_URI, "/", NEO4J_DATABASE)

dataset : .data/CIC-UNSW/CICFlowMeter_out.csv
neo4j   : neo4j+s://837e502f.databases.neo4j.io / neo4j


## Inspect the source data

The file is ~1.9 GB / 3.5 M rows, so we never load it whole into memory - we peek at a small slice here and **stream** it in chunks when loading.

In [ ]:
import pandas as pd

preview = pd.read_csv(DATASET_PATH, nrows=1000)
print(f"{preview.shape[1]} columns")
print("identity :", list(preview.columns[:7]))
print("label    :", preview.columns[-1])
print("features :", preview.shape[1] - 8, "statistical columns")
preview.head()

84 columns
identity : ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp']
label    : Label
features : 76 statistical columns


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,175.45.176.2-149.171.126.16-23357-80-6,175.45.176.2,23357,149.171.126.16,80,6,22/01/2015 07:50:15 AM,214392,9,21,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Exploits
1,175.45.176.0-149.171.126.16-13284-80-6,175.45.176.0,13284,149.171.126.16,80,6,22/01/2015 07:50:13 AM,2376792,9,3,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Reconnaissance
2,175.45.176.2-149.171.126.16-13792-5555-6,175.45.176.2,13792,149.171.126.16,5555,6,22/01/2015 07:50:16 AM,131350,10,3,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Exploits
3,175.45.176.0-149.171.126.15-39500-80-6,175.45.176.0,39500,149.171.126.15,80,6,22/01/2015 07:50:18 AM,164796,6,3,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DoS
4,175.45.176.0-149.171.126.14-29309-3000-6,175.45.176.0,29309,149.171.126.14,3000,6,22/01/2015 07:50:19 AM,163418,6,3,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Generic


## The graph model

Network-flow data is naturally a graph - hosts talk to hosts - but a flat `(:Host)-[:FLOW {…}]->(:Host)` edge model buries the most interesting dimensions
(service, protocol, attack class, time) inside relationship properties, where they cannot be indexed, shared, or traversed.

We instead **reify every flow as its own node**: a star / fact model with `:Flow` at the centre and shared *dimension* nodes around it. A flow is a fact; the host it came from, the host it went to, the port it targeted, the protocol it used, and the attack class it belongs to are all dimensions of that fact.

### Nodes

| Label | Key | Description | Cardinality (full load) |
|-------|-----|-------------|------------------------:|
| `:Host` | `ip` | A network endpoint (source or destination IP). | 40 src / 39 dst |
| `:Flow` | `key` | One flow record + its 76 CICFlowMeter features. | 3,540,241 |
| `:Port` | `number` | A destination / service port (e.g. 80 → HTTP). | 64,635 |
| `:Protocol` | `number` | L4 protocol (6 → TCP, 17 → UDP). | 2 |
| `:AttackCategory` | `name` | One of the 10 UNSW-NB15 classes. | 10 |
| `:Subnet` | `cidr` | The /24 a host belongs to (separates the attacker range `175.45.176.0/24` from the victim range `149.171.126.0/24`). | derived |

### Relationships

| Pattern | Meaning |
|---------|---------|
| `(:Host)-[:SENDS]->(:Flow)` | the source host originated the flow |
| `(:Flow)-[:TO]->(:Host)` | the flow was destined for this host |
| `(:Flow)-[:USES_PROTOCOL]->(:Protocol)` | transport protocol of the flow |
| `(:Flow)-[:ON_PORT]->(:Port)` | destination (service) port |
| `(:Flow)-[:CLASSIFIED_AS]->(:AttackCategory)` | ground-truth label |
| `(:Host)-[:IN_SUBNET]->(:Subnet)` | host membership of a /24 network |
| `(:Host)-[:CONNECTED_TO {flows, attackFlows, bytes}]->(:Host)` | **derived** aggregate edge summarising every flow between a host pair |

```mermaid
graph LR
    subnetA([":Subnet"]) -. IN_SUBNET .- src
    src([":Host (src)"]) -- SENDS --> flow[":Flow"]
    flow -- TO --> dst([":Host (dst)"])
    dst -. IN_SUBNET .- subnetB([":Subnet"])
    flow -- USES_PROTOCOL --> proto([":Protocol"])
    flow -- ON_PORT --> port([":Port"])
    flow -- CLASSIFIED_AS --> cat([":AttackCategory"])
    src -- "CONNECTED_TO (derived)" --> dst
```

### Design notes

* **Why reify the flow?** Dimension nodes are *shared*: every flow to port 80 hangs off
  the single `:Port {number: 80}` node, so "what is the most attacked service?" is a
  one-hop aggregation instead of a scan over relationship properties. New dimensions
  (e.g. a TLS-fingerprint node) can be added later without touching existing data.
* **Direction encodes semantics.** The path
  `(:Host)-[:SENDS]->(:Flow)-[:TO]->(:Host)` reads as *attacker → flow → victim*, so
  the natural traversal direction is the direction of the connection.
* **Source port is a property, not a node.** Ephemeral client ports are high-cardinality
  and analytically uninteresting, so `srcPort` lives on the `:Flow`. The *destination*
  port identifies the service and is shared, so it becomes a `:Port` node.
* **`label` is duplicated** onto the `:Flow` (for fast `WHERE f.label <> 'Benign'`
  filtering and indexing) *and* modelled as a `:CLASSIFIED_AS` edge (for traversal and
  per-category aggregation).
* **`timestamp`** is stored as a native Neo4j `datetime` so time-window queries and
  ordering work natively.
* **`:CONNECTED_TO`** is materialised after the bulk load (section 7). Only ~40×39 host
  pairs exist, so this collapses millions of flows into a compact host-communication
  graph that is ideal for centrality / community algorithms.

## Connect to Neo4j

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()


def run_write(query, **params):
    """Run a single auto-committed write against the configured database."""
    with driver.session(database=NEO4J_DATABASE) as session:
        return session.run(query, **params).consume()


print("connected to", NEO4J_URI, "→ database", NEO4J_DATABASE)

connected to neo4j+s://837e502f.databases.neo4j.io → database neo4j


## Schema: constraints & indexes

Uniqueness constraints are created **before** loading. Each one is backed by an index, which is what makes the `MERGE` calls in the loader fast (an index seek instead of a full label scan) and guarantees the load is **idempotent** - re-running it updates existing nodes rather than duplicating them.

In [ ]:
SCHEMA = [
    # uniqueness constraints (each creates a backing index)
    "CREATE CONSTRAINT host_ip        IF NOT EXISTS FOR (h:Host)           REQUIRE h.ip     IS UNIQUE",
    "CREATE CONSTRAINT flow_key       IF NOT EXISTS FOR (f:Flow)           REQUIRE f.key    IS UNIQUE",
    "CREATE CONSTRAINT port_number    IF NOT EXISTS FOR (p:Port)           REQUIRE p.number IS UNIQUE",
    "CREATE CONSTRAINT protocol_num   IF NOT EXISTS FOR (p:Protocol)       REQUIRE p.number IS UNIQUE",
    "CREATE CONSTRAINT attackcat_name IF NOT EXISTS FOR (a:AttackCategory) REQUIRE a.name   IS UNIQUE",
    "CREATE CONSTRAINT subnet_cidr    IF NOT EXISTS FOR (s:Subnet)         REQUIRE s.cidr   IS UNIQUE",
    # secondary indexes for common filters
    "CREATE INDEX flow_label IF NOT EXISTS FOR (f:Flow) ON (f.label)",
    "CREATE INDEX flow_ts    IF NOT EXISTS FOR (f:Flow) ON (f.timestamp)",
]

for stmt in SCHEMA:
    run_write(stmt)
print(f"{len(SCHEMA)} schema statements applied")

8 schema statements applied


## Reference / dimension nodes

The `:Protocol` and `:AttackCategory` dimensions are small, closed vocabularies, so we seed them up-front with friendly names. The numeric label codes come straight from `Readme.txt`. We also keep a small well-known-port → service map used to annotate `:Port` nodes during the load.

In [ ]:
# L4 protocol numbers seen in the data (and a few common extras for completeness)
PROTOCOLS = {0: "HOPOPT", 1: "ICMP", 2: "IGMP", 6: "TCP", 17: "UDP", 58: "IPv6-ICMP"}

# UNSW-NB15 label codes (from Readme.txt) -> name; everything except Benign is malicious
ATTACK_CATEGORIES = {
    "Benign": 0,
    "Analysis": 1,
    "Backdoor": 2,
    "DoS": 3,
    "Exploits": 4,
    "Fuzzers": 5,
    "Generic": 6,
    "Reconnaissance": 7,
    "Shellcode": 8,
    "Worms": 9,
}

# Well-known destination ports -> service label (annotation only)
WELL_KNOWN_PORTS = {
    20: "FTP-Data",
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    67: "DHCP",
    80: "HTTP",
    110: "POP3",
    111: "RPC",
    123: "NTP",
    135: "MS-RPC",
    139: "NetBIOS",
    143: "IMAP",
    161: "SNMP",
    179: "BGP",
    389: "LDAP",
    443: "HTTPS",
    445: "SMB",
    514: "Syslog",
    587: "SMTP-Sub",
    993: "IMAPS",
    995: "POP3S",
    1433: "MSSQL",
    1521: "Oracle",
    3306: "MySQL",
    3389: "RDP",
    5060: "SIP",
    5432: "PostgreSQL",
    5555: "Freeciv/ADB",
    6379: "Redis",
    8080: "HTTP-Alt",
    8443: "HTTPS-Alt",
}

run_write(
    "UNWIND $rows AS r MERGE (p:Protocol {number: r.number}) SET p.name = r.name",
    rows=[{"number": k, "name": v} for k, v in PROTOCOLS.items()],
)
run_write(
    """UNWIND $rows AS r
       MERGE (a:AttackCategory {name: r.name})
       SET a.id = r.id, a.malicious = (r.name <> 'Benign')""",
    rows=[{"name": k, "id": v} for k, v in ATTACK_CATEGORIES.items()],
)
print(
    "seeded",
    len(PROTOCOLS),
    "protocols and",
    len(ATTACK_CATEGORIES),
    "attack categories",
)

seeded 6 protocols and 10 attack categories


## Load the flows (batched streaming)

We stream `CICFlowMeter_out.csv` in chunks so memory stays flat regardless of file size. Each chunk is turned into a list of plain-Python dicts and sent to Neo4j with a single `UNWIND … MERGE` statement per batch.

A few data-hygiene details handled below:

* **Property names** - the 76 feature columns (`"Flow Bytes/s"`, `"Down/Up Ratio"`, …) are sanitised to camelCase Neo4j property keys (`flowBytesPerSec`, `downUpRatio`).
* **CICFlowMeter quirks** - `Infinity` / `NaN` values (common in the bytes/s columns for zero-duration flows) are converted to `null`; numpy scalars are cast to native Python types the driver can serialise.
* **Timestamp** - parsed from `dd/mm/YYYY hh:mm:ss AM/PM` and sent as ISO-8601 so `datetime()` stores it natively.
* **`key`** - a synthetic, monotonically increasing row id. The 5-tuple `Flow ID` recurs over time, so it is *not* unique; `key` is, which makes `MERGE (:Flow {key})` both correct and idempotent.

> **Tip:** set `MAX_ROWS` to a small number (e.g. `200_000`) for a quick smoke-test load; leave it `None` to load all 3.5 M flows.

In [ ]:
import re
import math
import numpy as np

ID_COLS = {
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP",
    "Dst Port",
    "Protocol",
    "Timestamp",
    "Label",
}


def to_prop(col):
    """'Flow Bytes/s' -> 'flowBytesPerSec', 'Down/Up Ratio' -> 'downUpRatio'."""
    s = col.strip().replace("/s", " per sec").replace("/", " ")
    parts = re.sub(r"[^0-9A-Za-z]+", " ", s).split()
    return parts[0].lower() + "".join(p.capitalize() for p in parts[1:])


def clean(v):
    """Make a value JSON/driver-safe: numpy -> python, NaN/Inf -> None."""
    if v is None:
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (np.floating, float)):
        f = float(v)
        return None if (math.isnan(f) or math.isinf(f)) else f
    return v


LOAD_FLOWS = """
UNWIND $rows AS row
MERGE (src:Host {ip: row.srcIp})
MERGE (dst:Host {ip: row.dstIp})
MERGE (proto:Protocol {number: row.protocol})
MERGE (port:Port {number: row.dstPort})
  ON CREATE SET port.service = row.service
MERGE (cat:AttackCategory {name: row.label})
MERGE (f:Flow {key: row.key})
SET f += row.props,
    f.flowId    = row.flowId,
    f.srcPort   = row.srcPort,
    f.dstPort   = row.dstPort,
    f.protocol  = row.protocol,
    f.label     = row.label,
    f.timestamp = CASE WHEN row.timestamp IS NULL THEN NULL ELSE datetime(row.timestamp) END
MERGE (src)-[:SENDS]->(f)
MERGE (f)-[:TO]->(dst)
MERGE (f)-[:USES_PROTOCOL]->(proto)
MERGE (f)-[:ON_PORT]->(port)
MERGE (f)-[:CLASSIFIED_AS]->(cat)
"""


def build_rows(chunk, prop_map, start_key):
    ts = pd.to_datetime(
        chunk["Timestamp"], format="%d/%m/%Y %I:%M:%S %p", errors="coerce"
    )
    records = chunk.to_dict("records")
    rows = []
    for i, rec in enumerate(records):
        t = ts.iloc[i]
        rows.append(
            {
                "key": int(start_key + i),
                "flowId": rec["Flow ID"],
                "srcIp": rec["Src IP"],
                "dstIp": rec["Dst IP"],
                "srcPort": int(rec["Src Port"]),
                "dstPort": int(rec["Dst Port"]),
                "protocol": int(rec["Protocol"]),
                "service": WELL_KNOWN_PORTS.get(int(rec["Dst Port"])),
                "label": rec["Label"],
                "timestamp": None if pd.isna(t) else t.isoformat(),
                "props": {prop_map[c]: clean(rec[c]) for c in prop_map},
            }
        )
    return rows

In [ ]:
from tqdm.auto import tqdm

BATCH = 5_000  # rows per write transaction
MAX_ROWS = None  # None -> load everything; or e.g. 200_000 for a quick test

prop_map = None
key = 0
loaded = 0

reader = pd.read_csv(DATASET_PATH, chunksize=BATCH)
with driver.session(database=NEO4J_DATABASE) as session:
    for chunk in tqdm(reader, unit="batch"):
        if MAX_ROWS is not None:
            if loaded >= MAX_ROWS:
                break
            if loaded + len(chunk) > MAX_ROWS:
                chunk = chunk.iloc[: MAX_ROWS - loaded]

        if prop_map is None:  # build the feature-column -> property-name map once
            prop_map = {c: to_prop(c) for c in chunk.columns if c not in ID_COLS}

        rows = build_rows(chunk, prop_map, key)
        session.execute_write(lambda tx: tx.run(LOAD_FLOWS, rows=rows).consume())
        key += len(rows)
        loaded += len(rows)

print(f"loaded {loaded:,} flows")

0batch [00:00, ?batch/s]

loaded 3,540,241 flows


## Derive subnets & host-to-host aggregate edges

Two post-processing steps that enrich the graph for analysis:

1. **`:Subnet`** - derive each host's /24 and link it. This cleanly separates the attacker range (`175.45.176.0/24`) from the victim range (`149.171.126.0/24`).
2. **`:CONNECTED_TO`** - collapse the millions of `:Flow` facts into one weighted edge per ordered host pair (`flows`, `attackFlows`, total `bytes`). This compact projection is what you would run centrality / community detection over.

In [ ]:
# 7a. Subnet membership (/24)
run_write("""
MATCH (h:Host)
WITH h, split(h.ip, '.') AS o
WITH h, o[0] + '.' + o[1] + '.' + o[2] + '.0/24' AS cidr
MERGE (s:Subnet {cidr: cidr})
MERGE (h)-[:IN_SUBNET]->(s)
""")

# 7b. Aggregate host-to-host communication edges
run_write("""
MATCH (src:Host)-[:SENDS]->(f:Flow)-[:TO]->(dst:Host)
WITH src, dst,
     count(f)                                                   AS flows,
     sum(CASE WHEN f.label <> 'Benign' THEN 1 ELSE 0 END)       AS attackFlows,
     sum(coalesce(f.totalLengthOfFwdPacket, 0.0)
       + coalesce(f.totalLengthOfBwdPacket, 0.0))               AS bytes
MERGE (src)-[c:CONNECTED_TO]->(dst)
SET c.flows = flows, c.attackFlows = attackFlows, c.bytes = bytes
""")

print("subnets and CONNECTED_TO edges derived")

subnets and CONNECTED_TO edges derived


## Validation & example queries

Sanity-check the load and show off a few questions the graph model answers in one or two hops.

In [ ]:
QUERIES = {
    "node counts by label": "MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY count DESC",
    "flows per attack category": """MATCH (f:Flow)-[:CLASSIFIED_AS]->(c:AttackCategory)
           RETURN c.name AS category, c.malicious AS malicious, count(f) AS flows
           ORDER BY flows DESC""",
    "top 10 targeted services (attacks only)": """MATCH (f:Flow)-[:ON_PORT]->(p:Port)
           WHERE f.label <> 'Benign'
           RETURN p.number AS port, p.service AS service, count(f) AS attackFlows
           ORDER BY attackFlows DESC LIMIT 10""",
    "noisiest attacker -> victim pairs": """MATCH (src:Host)-[c:CONNECTED_TO]->(dst:Host)
           WHERE c.attackFlows > 0
           RETURN src.ip AS attacker, dst.ip AS victim,
                  c.attackFlows AS attackFlows, c.flows AS totalFlows
           ORDER BY attackFlows DESC LIMIT 10""",
    "cross-subnet attack traffic": """MATCH (sa:Subnet)<-[:IN_SUBNET]-(:Host)-[c:CONNECTED_TO]->(:Host)-[:IN_SUBNET]->(da:Subnet)
           RETURN sa.cidr AS fromSubnet, da.cidr AS toSubnet,
                  sum(c.attackFlows) AS attackFlows
           ORDER BY attackFlows DESC""",
}

with driver.session(database=NEO4J_DATABASE) as session:
    for title, q in QUERIES.items():
        print(f"\n### {title}")
        for rec in session.run(q):
            print("   ", dict(rec))


### node counts by label
    {'label': 'Flow', 'count': 3540241}
    {'label': 'Port', 'count': 64635}
    {'label': 'Host', 'count': 43}
    {'label': 'AttackCategory', 'count': 10}
    {'label': 'Subnet', 'count': 8}
    {'label': 'Protocol', 'count': 6}

### flows per attack category
    {'category': 'Benign', 'malicious': False, 'flows': 3450658}
    {'category': 'Exploits', 'malicious': True, 'flows': 30951}
    {'category': 'Fuzzers', 'malicious': True, 'flows': 29613}
    {'category': 'Reconnaissance', 'malicious': True, 'flows': 16735}
    {'category': 'Generic', 'malicious': True, 'flows': 4632}
    {'category': 'DoS', 'malicious': True, 'flows': 4467}
    {'category': 'Shellcode', 'malicious': True, 'flows': 2102}
    {'category': 'Backdoor', 'malicious': True, 'flows': 452}
    {'category': 'Analysis', 'malicious': True, 'flows': 385}
    {'category': 'Worms', 'malicious': True, 'flows': 246}

### top 10 targeted services (attacks only)
    {'port': 80, 'service': 'HTTP', '

In [ ]:
# Close the driver when finished
driver.close()
print("driver closed")

driver closed
